# Smooth SR-USR-SR Transition Analysis

This notebook demonstrates the implementation of the **Smooth Slow-Roll to Ultra-Slow-Roll** inflationary model proposed in [arXiv:2603.17465v1](file:///C:/Users/diego/OneDrive/Documentos/Universidad/Cosmologia/A-NumInflation/SR-USR-SR_Transition/2603.17465v1%20(1).pdf).

## 1. Model Overview

Traditional USR models often feature sharp transitions in the second slow-roll parameter $\epsilon_2$, which leads to physically unnatural discontinuities. This model proposes a smooth transition by defining the effective mass index $\nu^2(\tau)$ as a polynomial function of conformal time:

$$ F(\tau) = \left( \mu^2 - \frac{9}{4} \right) - \alpha \left( \frac{\tau}{\tau_*} \right) + q^2 \left( \frac{\tau}{\tau_*} \right)^2 $$

Where the parameters are constrained by $\mu^2 = 9/4 + \alpha - q^2$ to ensure continuity at the transition point $\tau_*$.

## 2. Reconstructing the Potential

The model allows for analytical tracking of the background quantity $z(\tau)$. To integrate this into our numerical solver, we reconstruct the effective potential $V(\phi)$ by:
1. Calculating $\epsilon_1(\tau)$ from the analytical $z(\tau)$.
2. Mapping to e-folds $N$.
3. Integrating the field displacement $\Delta \phi = \int \sqrt{2\epsilon_1} dN$.
4. Parameterizing $V(\phi) \approx 3M_{pl}^2 H_0^2 (1 - \epsilon_1/3)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Ensure we can import from the parent directory
sys.path.append(os.path.abspath('..'))

from inflation_models import SmoothUSRTransitionModel
from inf_dyn_background import run_background_simulation, get_derived_quantities

# Instantiate the model with default paper parameters
model = SmoothUSRTransitionModel(alpha=22.63, mu=2.0294, eps_sr1=1e-6)

print("Model Initialized: ", model.name)

## 3. Visualizing Reconstructed Background

Let's look at the background parameters derived analytically during the model initialization.

In [ ]:
plt.figure(figsize=(12, 5))

# Plot Reconstructed Potential
plt.subplot(1, 2, 1)
plt.plot(model.phi_grid, model.V_grid)
plt.title('Reconstructed Potential $V(\phi)$')
plt.xlabel('$\phi / M_{pl}$')
plt.ylabel('$V / V_0$')
plt.grid(True)

# Demonstrate f(x) and derivatives
phi_test = np.linspace(model.phi_grid.min(), model.phi_grid.max(), 100)
plt.subplot(1, 2, 2)
plt.plot(phi_test, model.dfdx(phi_test), label="dV/dphi")
plt.title('Potential Derivatives')
plt.xlabel('$\phi / M_{pl}$')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 4. Exact Numerical Background Evolution

Now we run the exact dynamical equations through the `run_background_simulation` function to see if the numerical evolution follows the intended analytical path.

In [ ]:
# Define time span (delta T based on H/S approx)
T_span = np.linspace(0, 0.0015, 20000)

sol_data = run_background_simulation(model, T_span)
derived = get_derived_quantities(sol_data, model)

# Plot epsH(N)
plt.figure(figsize=(10, 6))
plt.plot(derived['N'], derived['epsH'], label='Numerical $\epsilon_H$')
plt.yscale('log')
plt.xlabel('e-folds $N$')
plt.ylabel('$\epsilon_H$')
plt.title('Numerical Slow-Roll Parameter Evolution')
plt.axhline(1, color='red', linestyle='--', label='End of Inflation')
plt.grid(True, which="both", ls="-")
plt.legend()
plt.show()

## 5. Conclusion

The numerical background simulation confirms that the reconstructed potential effectively captures the Smooth USR phase. The system smoothly transitions from the SR regime into the USR regime (where $\epsilon_H$ drops significantly) and back, without the discontinuities typically found in step-function models.